In [7]:
import librosa
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
import re
from itertools import product
import json
import scoreq
from pathlib import Path

dll_path = os.path.join(sys.prefix, 'Library', 'bin')
if os.path.exists(dll_path):
    os.add_dll_directory(dll_path)
    print("✅ FFmpeg DLLs linked.")

✅ FFmpeg DLLs linked.


Runs SCOREQ in no reference and non-matching reference mode on lufs_normalised and split (also lufs normalised) files.

Saves outputs to separate files in data/results.

In [8]:
project_dir = Path.cwd().parent

norm_path = project_dir / 'data' / 'trimmed' / 'lufs_normalised'
output_path = project_dir / 'data' / 'results' / 'scoreq_outputs_lufs.jsonl'
split_path = project_dir / 'data' / 'trimmed' / 'split'
split_output_path = project_dir / 'data' / 'results' / 'scoreq_outputs_split.jsonl'
refs_path = project_dir / 'data' / 'refs'

In [9]:
os.makedirs(output_path.parent, exist_ok=True)

In [10]:
# create model instances

model_nr = scoreq.Scoreq(data_domain='natural', mode='nr')
model_nmr = scoreq.Scoreq(data_domain='natural', mode='ref')

SCOREQ (ONNX) initialized on provider: CPUExecutionProvider
SCOREQ (ONNX) initialized on provider: CPUExecutionProvider


In [11]:
field_ref = os.path.join(refs_path, 'field_ref.wav')
garden_ref =  os.path.join(refs_path, 'garden_ref.wav')

In [12]:
# run both no reference and non-matching reference modes on audio
# split and non split results are saved to different files

print ('------------Original files------------')

with open(output_path, 'w', encoding='utf-8') as fout:
    for fin in os.listdir(norm_path):
        full_fname = os.path.join(norm_path, fin)

        if os.path.isdir(full_fname) or not fin.endswith('.wav'):
            continue
        
        pred_mos_nr = model_nr.predict(test_path=full_fname)
        if 'garden' in fin:
            pred_mos_nmr = model_nmr.predict(test_path=full_fname, ref_path=garden_ref)
        elif 'field' in fin:
            pred_mos_nmr = model_nmr.predict(test_path=full_fname, ref_path=field_ref)
            
        results = {'key': fin, 'scoreq_nr': pred_mos_nr, 'scoreq_nmr': pred_mos_nmr}
        
        json_line = json.dumps(results)
        
        fout.write(json_line + '\n')
        
        print(f'file: {fin} - predicted MOS (nr): {pred_mos_nr} | predicted MOS (nmr): {pred_mos_nmr}')

print('\n------------Split files------------')

with open(split_output_path, 'w', encoding='utf-8') as fout:
    for fin in os.listdir(split_path):
        full_fname = os.path.join(split_path, fin)
        
        pred_mos_nr = model_nr.predict(test_path=full_fname)
        if 'garden' in fin:
            pred_mos_nmr = model_nmr.predict(test_path=full_fname, ref_path=garden_ref)
        elif 'field' in fin:
            pred_mos_nmr = model_nmr.predict(test_path=full_fname, ref_path=field_ref)

        results = {'key': fin, 'scoreq_nr': pred_mos_nr, 'scoreq_nmr': pred_mos_nmr}
        
        json_line = json.dumps(results)
        
        fout.write(json_line + '\n')
        
        print(f'file: {fin} - predicted MOS (nr): {pred_mos_nr} | predicted MOS (nmr): {pred_mos_nmr}')

------------Original files------------
file: field1mflyby1_norm.wav - predicted MOS (nr): 0.9726290106773376 | predicted MOS (nmr): 0.8753518462181091
file: field1mflyby2_norm.wav - predicted MOS (nr): 1.0231995582580566 | predicted MOS (nmr): 0.8450523018836975
file: field1mhover1_norm.wav - predicted MOS (nr): 0.9046902060508728 | predicted MOS (nmr): 0.9557301998138428
file: field1mhover2_norm.wav - predicted MOS (nr): 0.9180837273597717 | predicted MOS (nmr): 0.9526504278182983
file: field3mflyby1_norm.wav - predicted MOS (nr): 1.009310007095337 | predicted MOS (nmr): 0.8037422895431519
file: field3mflyby2_norm.wav - predicted MOS (nr): 0.9863057732582092 | predicted MOS (nmr): 0.8346875905990601
file: field3mhover1_norm.wav - predicted MOS (nr): 0.9897759556770325 | predicted MOS (nmr): 0.9302586317062378
file: field3mhover2_norm.wav - predicted MOS (nr): 0.9715979695320129 | predicted MOS (nmr): 0.8825660943984985
file: field5mflyby1_norm.wav - predicted MOS (nr): 1.0145473480224